In [0]:
%run ./10_document_framework

## Ingestion and publication (reusable, operational)

Reads every UTF-8 `.md`/`.txt` file in `landing/` (PDF/DOCX/OCR parsing is deliberately excluded from this no-install version). Design decisions:

* **Eligibility comes from data, not files.** The active version is chosen by business date in the manifest; a late-arriving older file can never displace a newer accepted version, and a renamed file never creates a duplicate document.
* **Preflight the whole batch before admitting anything.** A different content for an existing `(document_id, version_id)`, or two versions claiming one business date, quarantines the incoming files (including a valid-looking sibling in the same ambiguous batch). A re-ingested identical file matches its stored hash, so reruns stay idempotent.
* **Chunks are written before version metadata**, so an interrupted run is repaired by rerunning this notebook.
* **Publication is a MERGE, not an overwrite**, so the active chunk table keeps its identity and history for a possible Delta Sync index later.

This notebook only **defines** `ingest_and_publish()`; it generates no files and has no side effects when `%run` from the demo or test notebooks. See the final cell for running an operational pass directly.

In [0]:
def read_source(path):
    """Read a landing file as UTF-8 text; parse/decode failures are quarantined upstream."""
    return path.read_text(encoding="utf-8")

def merge_rows(name, rows, schema, condition):
    """Insert-only MERGE: append rows whose match condition finds no target row.

    Existing matched rows are left untouched, which keeps every ingestion pass
    idempotent — reruns never duplicate or overwrite history.
    """
    if not rows: return
    src = spark.createDataFrame(rows, schema)
    src.createOrReplaceTempView("agri_merge_source")
    spark.sql(f"MERGE INTO {table(name)} t USING agri_merge_source s ON {condition} "
             "WHEN NOT MATCHED THEN INSERT *")

def publish_active_chunks():
    """Refresh active_document_chunks to exactly the registry's current chunks.

    A MERGE (not an overwrite) keeps the table's identity and history intact for
    a possible Delta Sync index later, updates/inserts current chunks, and
    physically deletes retired-version chunks from the retrieval source while
    document_chunks preserves the full history.
    """
    # MERGE preserves the table identity for Delta Sync and propagates deletions.
    source = spark.table(table("document_chunks")).alias("c").join(
        spark.table(table("active_document_registry")).alias("r"),
        (F.col("c.document_id")==F.col("r.document_id")) &
        (F.col("c.version_id")==F.col("r.version_id")) &
        (F.col("c.content_hash")==F.col("r.content_hash")), "inner").select("c.*")
    source.createOrReplaceTempView("agri_active_source")
    spark.sql(f"""MERGE INTO {table('active_document_chunks')} t
      USING agri_active_source s ON t.chunk_id = s.chunk_id
      WHEN MATCHED THEN UPDATE SET *
      WHEN NOT MATCHED THEN INSERT *
      WHEN NOT MATCHED BY SOURCE THEN DELETE""")

def ingest_and_publish():
    """Run one idempotent ingestion pass over landing/.

    Parses every UTF-8 .md/.txt file, preflights the whole batch against accepted
    history, quarantines rejects with a reason, persists accepted chunks and
    versions (chunks first), then publishes active chunks. Returns
    {'valid_files': int, 'quarantined_files': int} for this pass.
    """
    with LOCK:
        known = [r.asDict() for r in spark.table(table("document_versions")).collect()]
        parsed, quarantine = [], []
        now = datetime.now(timezone.utc).isoformat()
        for path in sorted(LANDING.iterdir()):
            if path.suffix.lower() not in (".md", ".txt"): continue
            raw_hash = digest(path.read_bytes().hex())
            try:
                rec = parse_document(read_source(path))
                parsed.append(dict(rec, source_path=str(path), filename=path.name, ingested_at=now))
            except Exception as exc:
                quarantine.append(dict(source_path=str(path), content_hash=raw_hash, reason=str(exc), observed_at=now))
        # Preflight the ENTIRE batch (incoming files plus accepted history)
        # before admitting anything: a different content for an existing
        # (document_id, version_id), or two versions claiming one business
        # date, quarantines the incoming files. A re-ingested identical file
        # matches its stored hash, so reruns stay idempotent.
        all_rows = known + parsed
        by_version, by_date = {}, {}
        for rec in all_rows:
            by_version.setdefault((rec['document_id'], rec['version_id']), set()).add(rec['content_hash'])
            by_date.setdefault((rec['document_id'], rec['document_date']), set()).add(rec['version_id'])
        valid = {}
        for rec in parsed:
            conflict = len(by_version[(rec['document_id'],rec['version_id'])]) > 1
            tie = len(by_date[(rec['document_id'],rec['document_date'])]) > 1
            if conflict or tie:
                quarantine.append(dict(source_path=rec['source_path'], content_hash=rec['content_hash'],
                                       reason='Conflicting version content' if conflict else 'Ambiguous document date', observed_at=now))
            else:
                valid.setdefault((rec['document_id'],rec['version_id']), rec)
        records = list(valid.values())
        # Write chunks BEFORE version metadata: if the run is interrupted,
        # rerunning this function repairs the partially published version idempotently.
        chunks = [c for rec in records for c in chunk_document(rec)]
        merge_rows("document_chunks", chunks, CHUNK_SCHEMA, "t.chunk_id = s.chunk_id")
        merge_rows("document_versions", records, VERSION_SCHEMA,
                   "t.document_id = s.document_id AND t.version_id = s.version_id")
        merge_rows("quarantined_documents", quarantine,
                   QUARANTINE_SCHEMA,
                   "t.source_path = s.source_path AND t.content_hash = s.content_hash AND t.reason = s.reason")
        publish_active_chunks()
        return dict(valid_files=len(records), quarantined_files=len(quarantine))

In [0]:
# --- Operational entry point -------------------------------------------------------
# When this notebook is %run from the demo or robustness notebooks, it only
# DEFINES ingest_and_publish() — the cell above has no side effects. To run a
# one-off operational ingestion pass directly from this notebook, set
# AUTO_RUN_INGESTION = True and run top to bottom. For scheduled ingestion, run
# this notebook with AUTO_RUN_INGESTION = True as a job task (or call
# ingest_and_publish() from a thin driver notebook) — never include the demo or
# robustness notebooks in an operational job.
AUTO_RUN_INGESTION = False
if AUTO_RUN_INGESTION:
    print(ingest_and_publish())
    display(spark.table(table("document_manifest")).orderBy("document_id", "document_date"))